In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

os.chdir(os.path.dirname(os.getcwd()))

from src.constants.config import path_portfolio, path_db
from src.constants.tickers import isin_to_ticker, currencies

In [ ]:
def search_database(path_db):
    """Returns Delisted stock from manual history db"""
    csv_files = [
        pd.read_csv(os.path.join(path_db, file), sep=';')
        for file in os.listdir(path_db)
        if file.endswith('.csv')
    ]
    return pd.concat(csv_files, ignore_index=True)


def collect_stock_rates(df):
    """Fetch historical stock data for a grouped DataFrame of tickers and return combined results."""
    all_data = []
    empty_data = []

    for _, row in df.iterrows():
        start_date = row['StartDate']
        end_date = row['EndDate']
        symbol = row['Ticker']
        product = row['Product']
        isin_code = row['ISIN']

        try:
            ticker = yf.Ticker(symbol)
            currency = ticker.info['currency']
            history_df = (
                ticker.history(start=start_date, end=end_date)
                .reset_index()[['Date', 'Close']]
                .assign(
                    Product=product,
                    ISIN=isin_code,
                    Ticker=symbol,
                    currency=currency
                )
            )
            print(f"Retrieved {history_df.shape[0]} rows for {symbol}")
            all_data.append(history_df)

        except Exception as error:
            print(f"Error fetching data for {symbol}: {error}")
            empty_data.append({
                "Ticker": symbol,
                "Product": product,
                "ISIN": isin_code,
                "StartDate": start_date,
                "EndDate": end_date,
                "Error": str(error)
            })

    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df['Date'] = pd.to_datetime(combined_df['Date'].astype(str).str.slice(start=0,stop=10)).dt.date

    empty_data = pd.DataFrame(empty_data)
    
    # read db
    print(f'loading delisted stocks from db:{empty_data['Ticker'].unique()}')
    df_db = search_database(path_db)
    df_db = df_db.loc[df_db['Ticker'].isin(empty_data['Ticker'].unique())]

    df = pd.concat([combined_df, df_db])
    return df


def collect_currency_data(currencies):
    """
    Fetch historical currency data using yfinance and return a combined DataFrame.

    Parameters:
        currencies (list): List of currency tickers.

    Returns:
        pd.DataFrame: Combined DataFrame with historical FX rates.
    """
    currency_data = []

    for currency in currencies:
        print(f"Fetching currency data for: {currency}")

        try:
            ticker = yf.Ticker(currency)
            cur_df = (
                ticker.history(period='max')
                .reset_index()[['Date', 'Close']]
                .assign(Product=currency)
                .assign(currency=currency.replace('=X', '').replace('EUR', ''))
                .rename(columns={'Close': 'Fx_rate'})
            )

            print(f"Retrieved {cur_df.shape[0]} rows for {currency}")
            currency_data.append(cur_df)

        except Exception as error:
            print(f"Error fetching data for {currency}: {error}")

    combined_df = pd.concat(currency_data, ignore_index=True)
    combined_df['Date'] = pd.to_datetime(combined_df['Date']).dt.date

    return combined_df


In [ ]:
# load portfolio
df_portfolio = pd.read_csv(path_portfolio)

df_portfolio['Ticker'] = df_portfolio['ISIN'].replace(isin_to_ticker)
df_portfolio['SnapshotDate'] = pd.to_datetime(df_portfolio['SnapshotDate']).dt.date
df_portfolio = df_portfolio.rename({'SnapshotDate':'Date'},axis=1)

df_grouped = df_portfolio.groupby(['Product', 'ISIN', 'Ticker']).agg({'Date': ['min', 'max']})
df_grouped.columns = ['StartDate', 'EndDate']
df_grouped = df_grouped.reset_index()

In [ ]:
# Collect stocks
df_stock_rates  = collect_stock_rates(df_grouped)

# Collect exchange rates
df_curr = collect_currency_data(currencies)

In [ ]:
df_stock_rates

In [ ]:
df = df_portfolio.merge(df_stock_rates[['Date','Ticker','Close','currency']],
                  on=['Date','Ticker'],
                  how='inner').merge(df_curr[['Date','currency','Fx_rate']],
                                     on=['Date','currency'],
                                     how='left')

df['Fx_rate'] = np.where(df['currency']=='EUR',1,df['Fx_rate'])

In [ ]:
df['eur'] = df['Close']/df['Fx_rate']*df['Aantal']